# Opion RL iteration — tethered PPO update (T4 GPU)

**Per iteration:** drag-drop `rl_iter.tar` (from the chat) → run all cells →
download `policy_np.npz` + `policy_rl.pt` (+ read RESULT) → give back.

- Data: real-engine rollouts (student vs frozen pool). ctx[14]=0 everywhere.
- Loss: PPO clip + value MSE + entropy + **tether** KL(p_soft ‖ π) on a fixed
  packed-state probe (the leash to the teacher; β from beta.txt).
- GAE within each (pid,hand) decision chain, γ=1, λ=0.95; terminal = hand net.

In [ ]:
# 1. Runtime check
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "set Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 2. Keepalive
import threading, time, subprocess

def _ka():
    while True:
        try:
            out = subprocess.run(["free", "-h"], capture_output=True, text=True).stdoutq
            print("[keepalive]\n" + out, flush=True)
        except Exception as e:
            print("[keepalive] err", e, flush=True)
        time.sleep(300)

threading.Thread(target=_ka, daemon=True).start()
print("keepalive running")

In [ ]:
# 3. Packed data for the tether probe (first run only)
import glob, os, tarfile
if not glob.glob("packed/*/*/acts.npz"):
    if not os.path.exists("packed.tar"):
        import gdown
        gdown.download(id="1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA", output="packed.tar")
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")
print(len(glob.glob("packed/*/*/acts.npz")), "shards")

In [ ]:
# 4. Drag-drop rl_iter.tar here (rollouts + beta.txt + np_policy.py)
import os
assert os.path.exists("rl_iter.tar"), "upload rl_iter.tar first"
import tarfile
with tarfile.open("rl_iter.tar") as tf:
    tf.extractall("/content")
print(sorted(os.listdir("rl_iter")))

In [ ]:
# 5. Model: actor + value head; init from previous iteration
import torch
import torch.nn as nn

N_TOK, N_CTX, N_ACT, VOCAB = 41, 17, 6, 85


def _map_tokens(t):
    t = t.long()
    return torch.where(t < 53, t, 52 + t.clamp(min=1, max=32))


class Block(nn.Module):
    def __init__(self, d, h, f):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.h = h
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, f)
        self.fc2 = nn.Linear(f, d)
        self.act = nn.GELU()

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(B, T, self.h, -1).transpose(1, 2)
        k = k.view(B, T, self.h, -1).transpose(1, 2)
        v = v.view(B, T, self.h, -1).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), -1)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.fc2(self.act(self.fc1(self.ln2(x))))
        return x


class PPOPolicy(nn.Module):
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX), nn.Linear(N_CTX, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, N_ACT)
        self.value = nn.Linear(d, 1)

    def trunk(self, tokens, ctx):
        x = self.emb(_map_tokens(tokens)) + self.pos.weight[None, :N_TOK]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.ln_f(x).mean(dim=1)

    def forward(self, tokens, ctx):
        return self.head(self.trunk(tokens, ctx))

    def forward_both(self, tokens, ctx):
        p = self.trunk(tokens, ctx)
        return self.head(p), self.value(p).squeeze(-1)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


dev = "cuda"
model = PPOPolicy().to(dev)
INIT = "rl_iter/policy_rl.pt" if os.path.exists("rl_iter/policy_rl.pt")     else "rl_iter/final_act_v2.pt"
ck = torch.load(INIT, map_location="cpu")
sd = ck["model"] if "model" in ck else ck
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f"init from {INIT} | missing={list(missing)} unexpected={list(unexpected)}")
assert not unexpected, "bad checkpoint"
print("params:", f"{model.n_params():,}")

In [ ]:
# 6. Diff gate: torch vs numpy port on 2,048 packed states (every run)
import sys
sys.path.insert(0, "/content/rl_iter")
import numpy as np
import np_policy

z0 = np.load(sorted(glob.glob("packed/*/*/acts.npz"))[0], allow_pickle=False)
sel = np.flatnonzero(z0["part"] == 0)[:2048]
tk = z0["tokens"][sel].astype(np.int64)
cx = z0["ctx"][sel].astype(np.float32)
cx[:, 14] = 0.0
with torch.no_grad():
    ref = model.forward(torch.from_numpy(tk).cuda(),
                        torch.from_numpy(cx).cuda()).cpu().numpy()
pol = np_policy.NumpyPolicy({k: v.detach().cpu().numpy()
                             for k, v in model.state_dict().items()
                             if not k.startswith("value.")})
port = np.concatenate([pol.forward(tk[i:i+1024], cx[i:i+1024])
                       for i in range(0, len(tk), 1024)])
dmax = float(np.abs(ref.astype(np.float64) - port.astype(np.float64)).max())
print(f"diff gate: max|dlogit|={dmax:.2e}")
assert dmax < 1e-3, "numpy port gate FAILED"
print("GATE_PASS")

In [ ]:
# 7. Load rollouts -> per-(pid,hand) chains -> GAE advantages
import json

fs = sorted(glob.glob("rl_iter/rollouts/*.npz"))
assert fs, "no rollouts in tar"
cols = {k: [] for k in ("tokens", "ctx", "legal", "bucket", "logp",
                        "pid", "hand", "seq", "street", "reward")}
for f in fs:
    z = np.load(f)
    for k in cols:
        cols[k].append(z[k])
D = {k: np.concatenate(v) for k, v in cols.items()}
D["ctx"][:, 14] = 0.0
print(f"rollouts: {len(fs)} shards, {len(D['bucket']):,} decisions, "
      f"{len(set(zip(D['pid'].tolist(), D['hand'].tolist()))):,} chains")

order = np.lexsort((D["seq"], D["hand"], D["pid"]))
for k in D:
    D[k] = D[k][order]

chain_id, start, cid = [], 0, 0
pid, hnd = D["pid"], D["hand"]
same = (pid[1:] == pid[:-1]) & (hnd[1:] == hnd[:-1])
chain = np.concatenate([[0], np.cumsum(~same)])
n_chains = int(chain[-1]) + 1

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    vals = []
    for i in range(0, len(D["bucket"]), 16384):
        sl = slice(i, i + 16384)
        _, v = model.forward_both(torch.from_numpy(D["tokens"][sl]).cuda(),
                                  torch.from_numpy(D["ctx"][sl]).cuda())
        vals.append(v.float())
    V = torch.cat(vals).cpu().numpy().astype(np.float64)

R = np.where(np.concatenate([same, [False]]) == False, D["reward"], 0.0)
is_last = np.concatenate([same, [False]]) == False
R = np.where(is_last, D["reward"], 0.0)   # terminal row of each chain = hand net

adv = np.zeros_like(V)
lastgae = 0.0
for t in range(len(V) - 1, -1, -1):
    cont = t < len(V) - 1 and bool(same[t])
    nextv = V[t + 1] if cont else 0.0
    delta = R[t] + nextv - V[t]
    lastgae = delta + 0.95 * lastgae if cont else delta
    adv[t] = lastgae
ret = adv + V
adv = (adv - adv.mean()) / (adv.std() + 1e-8)
print(f"GAE done: {n_chains} chains, adv mean {adv.mean():.3f} sd {adv.std():.3f}")
print(f"reward/hand overall mean {D['reward'][is_last].mean():+.4f}")
mix = np.bincount(D["bucket"], minlength=6) / len(D["bucket"])
print("behavior bucket mix:", np.round(mix, 3).tolist())

In [ ]:
# 8. Tether probe: fixed 32,768 packed train states (ctx14=0)
probe_t, probe_c, probe_p = [], [], []
need = 32768
for f in sorted(glob.glob("packed/*/*/acts.npz")):
    z = np.load(f, allow_pickle=False)
    s = np.flatnonzero(z["part"] == 0)
    take = min(len(s), need - sum(len(x) for x in probe_t))
    if take <= 0:
        break
    s = s[:take]
    probe_t.append(z["tokens"][s])
    c = z["ctx"][s].astype(np.float32)
    c[:, 14] = 0.0
    probe_c.append(c)
    probe_p.append(z["p_soft"][s])
PT = torch.from_numpy(np.concatenate(probe_t)[:need].astype(np.int64)).cuda()
PC = torch.from_numpy(np.concatenate(probe_c)[:need].astype(np.float32)).cuda()
PP = torch.from_numpy(np.concatenate(probe_p)[:need].astype(np.float32)).cuda()
print("probe:", len(PT), "states")
BETA = float(open("rl_iter/beta.txt").read().strip())
print("beta =", BETA)

In [ ]:
# 9. PPO update: 2 epochs, fp16 autocast, minibatch 4096, clip 0.2 + tether
import time

EPOCHS, MB, CLIP, LR = 2, 4096, 0.2, 2e-5
ENT, VCOEF = 0.01, 0.5
N = len(D["bucket"])
T = torch.from_numpy(D["tokens"].astype(np.int64)).cuda()
C = torch.from_numpy(D["ctx"].astype(np.float32)).cuda()
Lg = torch.from_numpy(D["legal"].astype(np.float32)).cuda()
Bo = torch.from_numpy(D["bucket"].astype(np.int64)).cuda()
Lo = torch.from_numpy(D["logp"].astype(np.float32)).cuda()
Ad = torch.from_numpy(adv.astype(np.float32)).cuda()
Rt = torch.from_numpy(ret.astype(np.float32)).cuda()

opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")
idx_all = torch.randperm(N, device=dev)
t0 = time.time()
step = 0
for ep in range(EPOCHS):
    for i in range(0, N, MB):
        mb = idx_all[i:i + MB]
        with torch.autocast("cuda", dtype=torch.float16):
            logits, v = model.forward_both(T[mb], C[mb])
        logits = logits.float().masked_fill(Lg[mb] == 0, -1e9)
        logp = torch.log_softmax(logits, -1)
        logp_b = logp.gather(1, Bo[mb, None]).squeeze(1)
        ratio = torch.exp(logp_b - Lo[mb])
        s1 = ratio * Ad[mb]
        s2 = torch.clamp(ratio, 1 - CLIP, 1 + CLIP) * Ad[mb]
        pl = -torch.min(s1, s2).mean()
        vl = 0.5 * ((v.float() - Rt[mb]) ** 2).mean()
        q = torch.softmax(logits, -1)
        ent = -(q * torch.nan_to_num(logp, neginf=0.0)).sum(-1).mean()
        ps = slice((step * 4096) % (len(PT) - 4096),
                   (step * 4096) % (len(PT) - 4096) + 4096)
        with torch.autocast("cuda", dtype=torch.float16):
            plogits = model(PT[ps], PC[ps])
        pq = torch.log_softmax(plogits.float(), -1)
        kl = (PP[ps] * (torch.log(PP[ps] + 1e-8) - pq)).sum(-1).mean()
        loss = pl + VCOEF * vl - ENT * ent + BETA * kl
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        step += 1
        if step % 5 == 0:
            print(f"  step {step}/{((N + MB - 1) // MB) * EPOCHS} "
                  f"pl={pl.item():+.4f} v={vl.item():.4f} "
                  f"kl={kl.item():.4f} {time.time() - t0:.0f}s", flush=True)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    plogits = model(PT, PC).float()
pq = torch.log_softmax(plogits, -1)
kl_probe = float((PP * (torch.log(PP + 1e-8) - pq)).sum(-1).mean())
pmix = np.bincount(plogits.argmax(-1).cpu().numpy(), minlength=6) / len(PT)
print(f"post-update probe KL(p||pi)={kl_probe:.4f}")
print("probe argmax mix:", np.round(pmix, 3).tolist())
print("teacher ref      : [0.17, 0.44, 0.10, 0.26, 0.04, 0.00]")

In [ ]:
# 10. Save + export numpy weights + download
import json as _json
KL_STAR = 0.10
new_beta = min(10.0, max(0.01, BETA * float(np.exp(2 * (kl_probe - KL_STAR) / KL_STAR))))
RESULT = {"kl_probe": round(kl_probe, 4), "beta_next": round(new_beta, 4),
          "decisions": int(N), "epochs": EPOCHS, "lr": LR,
          "wall_min": round((time.time() - t0) / 60, 1),
          "probe_mix": [round(float(x), 4) for x in pmix]}
print("RESULT " + _json.dumps(RESULT))
torch.save({"model": model.state_dict(), "result": RESULT}, "policy_rl.pt")
np_policy.save_npz({k: v.detach().cpu().numpy()
                    for k, v in model.state_dict().items()}, "policy_np.npz")
from google.colab import files
files.download("policy_np.npz")
files.download("policy_rl.pt")
print("PPO_DONE")